# Notebook 3 — ResNet50 Baseline (Two-Phase Transfer Learning)

Trains a ResNet50 (pretrained on ImageNet) glaucoma classifier in two
phases -- Phase A (frozen backbone, linear-probe the FC head) followed by
Phase B (fine-tune `layer3`, `layer4`, `fc` at a lower backbone LR) -- and
evaluates the better of the two resulting checkpoints on the held-out test
set **exactly once**.

**This notebook consumes, and does not modify:**
- `split_manifest.csv` (Notebook 1's authoritative split)
- the materialized `dataset/{split}/{class}/` folders (Notebook 1)
- `common/preprocessing.py` (Notebook 2's CLAHE + transform + Dataset/DataLoader code)

**Official dataset (approved):** 2,468 usable images -- 1,925 Normal / 543
Glaucoma. Train 1,729 / Val 370 / Test 369. Seed 42.

**Model-selection rule (confirmed with the user before building this
notebook):** the "final selected ResNet50" evaluated on the test set is
**whichever of Phase A's best checkpoint and Phase B's best checkpoint has
the lower validation loss** -- not automatically Phase B. This is a
validation-set-only comparison (the test set is never touched for this
decision) and exists because unfreezing `layer3`+`layer4` (the majority of
ResNet50's parameters) against only 1,729 training images is a real
overfitting risk, not a hypothetical one -- Phase B is not guaranteed to
beat Phase A.

**No target accuracy/AUC is assumed anywhere in this notebook.** Whatever
the model actually achieves on the test set is what gets reported.

## 1. Environment setup and package installation

In [ ]:
import importlib, subprocess, sys

REQUIRED_PACKAGES = ["pandas", "numpy", "Pillow", "matplotlib", "opencv-python",
                      "torch", "torchvision", "scikit-learn"]

def _ensure(pkg):
    import_name = {"Pillow": "PIL", "opencv-python": "cv2", "scikit-learn": "sklearn"}.get(pkg, pkg)
    try:
        importlib.import_module(import_name)
        print(f"[ok]      {pkg} already available")
    except ImportError:
        print(f"[install] {pkg} not found, attempting pip install ...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
            print(f"[ok]      {pkg} installed")
        except Exception as e:
            print(f"[warning] could not install {pkg} automatically ({e}). "
                  f"Install it manually if a later cell fails on import.")

for _pkg in REQUIRED_PACKAGES:
    _ensure(_pkg)

import os, sys, json, hashlib, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, confusion_matrix, roc_curve)

print()
print("Environment ready.")
print("torch", torch.__version__, "| torchvision", __import__("torchvision").__version__,
      "| sklearn", __import__("sklearn").__version__)

## 2. Configuration, reproducibility, and device selection

Seed 42 applied to `random`, `numpy`, and `torch` (CPU and CUDA). `cudnn`
is set to deterministic mode -- this makes convolution algorithm selection
reproducible run-to-run at some performance cost, which is an explicit
trade-off in favor of reproducibility over raw speed, consistent with this
project's emphasis on reproducible results throughout Notebooks 1 and 2.

Device selection targets **CUDA (Colab T4) or CPU** -- not Apple Silicon
`mps` -- since the stated GPU target for real training is Colab's T4, and
sticking to `cuda`/`cpu` keeps behavior predictable and identical between a
local Mac (CPU, for pipeline checks only) and Colab (GPU, for real
training).

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Reproducibility seed set to {SEED} (random, numpy, torch, cuda).")
print("cudnn.deterministic =", torch.backends.cudnn.deterministic, "| cudnn.benchmark =", torch.backends.cudnn.benchmark)
print()
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU detected. Full training (15 + 20 epochs over 1,729 images) will be very slow on CPU.")
    print("This is expected when checking the pipeline locally -- run the real training in Colab with a T4 GPU.")

# --- Training hyperparameters (exactly as specified) ---
BATCH_SIZE = 32
PHASE_A_MAX_EPOCHS = 15
PHASE_A_LR = 1e-3
PHASE_B_MAX_EPOCHS = 20
PHASE_B_BACKBONE_LR = 1e-4
PHASE_B_FC_LR = 1e-3
SCHEDULER_PATIENCE = 3
SCHEDULER_FACTOR = 0.5
EARLY_STOP_PATIENCE = 6

# --- Class mapping (fixed throughout the project) ---
CLASS_MAPPING = {0: "Normal", 1: "Glaucoma"}
POSITIVE_CLASS = 1  # Glaucoma -- the clinically relevant "positive" class for
                     # precision/recall/sensitivity/specificity/ROC-AUC below.

print("\nCLASS_MAPPING:", CLASS_MAPPING, "| POSITIVE_CLASS:", POSITIVE_CLASS, "(Glaucoma)")
print(f"BATCH_SIZE={BATCH_SIZE} | Phase A: max_epochs={PHASE_A_MAX_EPOCHS}, lr={PHASE_A_LR}")
print(f"Phase B: max_epochs={PHASE_B_MAX_EPOCHS}, backbone_lr={PHASE_B_BACKBONE_LR}, fc_lr={PHASE_B_FC_LR}")
print(f"Scheduler: ReduceLROnPlateau(patience={SCHEDULER_PATIENCE}, factor={SCHEDULER_FACTOR}) monitoring val_loss")
print(f"Early stopping: patience={EARLY_STOP_PATIENCE} monitoring val_loss (same metric as checkpoint selection)")

## 3. Dataset/checkpoint path configuration

Same `RUNTIME_ENV` switch as Notebooks 1 and 2. `CHECKPOINT_DIR` is new --
where this notebook saves `.pt` checkpoints. In Colab, the Drive-mount cell
below runs automatically (and is a no-op outside Colab).

In [ ]:
RUNTIME_ENV = "local_mac"   # "local_mac" or "colab_drive"

PATH_PRESETS = {
    "local_mac": {
        "RAW_DATASET_ROOT": os.path.expanduser("~/Downloads/archive"),
        "OUTPUT_ROOT": os.path.expanduser("~/Downloads/GlaucomaProject"),
    },
    "colab_drive": {
        "RAW_DATASET_ROOT": "/content/drive/MyDrive/GlaucomaProject/raw_kaggle_download",
        "OUTPUT_ROOT": "/content/drive/MyDrive/GlaucomaProject",
    },
}

_paths = PATH_PRESETS[RUNTIME_ENV]
OUTPUT_ROOT = _paths["OUTPUT_ROOT"]

DATASET_OUT_DIR = os.path.join(OUTPUT_ROOT, "dataset")
RESULTS_OUT_DIR = os.path.join(OUTPUT_ROOT, "results")
COMMON_DIR = os.path.join(OUTPUT_ROOT, "common")
FIGURES_OUT_DIR = os.path.join(OUTPUT_ROOT, "figures")
CHECKPOINT_DIR = os.path.join(OUTPUT_ROOT, "checkpoints")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FIGURES_OUT_DIR, exist_ok=True)

PHASE_A_CKPT = os.path.join(CHECKPOINT_DIR, "resnet50_phaseA_best.pt")
PHASE_B_CKPT = os.path.join(CHECKPOINT_DIR, "resnet50_phaseB_best.pt")
FINAL_CKPT = os.path.join(CHECKPOINT_DIR, "resnet50_final_best.pt")

split_manifest_path = os.path.join(RESULTS_OUT_DIR, "split_manifest.csv")

print("RUNTIME_ENV     :", RUNTIME_ENV)
print("OUTPUT_ROOT     :", OUTPUT_ROOT)
print("DATASET_OUT_DIR :", DATASET_OUT_DIR)
print("RESULTS_OUT_DIR :", RESULTS_OUT_DIR)
print("COMMON_DIR      :", COMMON_DIR)
print("FIGURES_OUT_DIR :", FIGURES_OUT_DIR)
print("CHECKPOINT_DIR  :", CHECKPOINT_DIR, "(created)")

### 3b. Google Drive mount (Colab only, no-op elsewhere)

Only runs when this notebook is actually executing inside Google Colab
(detected via `google.colab` being importable). Does nothing on a local
Mac.

In [ ]:
try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted (Colab detected).")
except ImportError:
    print("Not running in Colab -- skipping Drive mount (this is expected on a local Mac).")

## 4. Pre-flight verification

Before touching any data or model weights, verify every precondition this
notebook depends on: required packages, GPU/CUDA visibility, dataset paths
from Notebooks 1 and 2, the class mapping, the `preprocessing` module, and
the checkpoint directory. Fails loudly (via `assert`) rather than partially
running if something upstream is missing.

In [ ]:
print("=== Pre-flight checks ===\n")

# 1. Packages
print("[1/6] Packages:")
for _pkg, _imp in [("torch", "torch"), ("torchvision", "torchvision"), ("cv2", "cv2"),
                    ("pandas", "pandas"), ("numpy", "numpy"), ("sklearn", "sklearn"),
                    ("matplotlib", "matplotlib"), ("PIL", "PIL")]:
    importlib.import_module(_imp)
    print(f"    OK  {_pkg}")

# 2. GPU / CUDA
print("\n[2/6] GPU/CUDA:")
print(f"    CUDA available: {torch.cuda.is_available()}")
print(f"    Selected device: {device}")
if not torch.cuda.is_available():
    print("    NOTE: no GPU visible in this environment -- fine for pipeline verification, "
          "required for full-scale training. Switch to a Colab T4 runtime for real training.")

# 3. Dataset paths from Notebook 1
print("\n[3/6] Dataset paths (Notebook 1 outputs):")
assert os.path.isfile(split_manifest_path), f"split_manifest.csv not found at {split_manifest_path}. Run Notebook 1 first."
assert os.path.isdir(DATASET_OUT_DIR), f"DATASET_OUT_DIR not found at {DATASET_OUT_DIR}. Run Notebook 1 first."
print(f"    OK  split_manifest.csv -> {split_manifest_path}")
print(f"    OK  DATASET_OUT_DIR    -> {DATASET_OUT_DIR}")

# 4. Class mapping consistency
print("\n[4/6] Class mapping:")
print(f"    CLASS_MAPPING = {CLASS_MAPPING}")
assert CLASS_MAPPING == {0: "Normal", 1: "Glaucoma"}, "Unexpected class mapping."
print("    OK  matches the fixed project convention (0=Normal, 1=Glaucoma).")

# 5. preprocessing module (Notebook 2 output)
print("\n[5/6] preprocessing module (Notebook 2 output):")
preprocessing_path = os.path.join(COMMON_DIR, "preprocessing.py")
assert os.path.isfile(preprocessing_path), (
    f"common/preprocessing.py not found at {preprocessing_path}. Run Notebook 2 first."
)
if COMMON_DIR not in sys.path:
    sys.path.insert(0, COMMON_DIR)
import preprocessing as pp
for _name in ["build_train_transform", "build_eval_transform", "GlaucomaDataset",
              "get_dataloaders", "compute_class_weights", "denormalize"]:
    assert hasattr(pp, _name), f"preprocessing.py is missing expected symbol: {_name}"
print(f"    OK  {preprocessing_path}")
print("    OK  all expected symbols present:", ["build_train_transform", "build_eval_transform",
      "GlaucomaDataset", "get_dataloaders", "compute_class_weights", "denormalize"])

# 6. Checkpoint directory
print("\n[6/6] Checkpoint directory:")
assert os.path.isdir(CHECKPOINT_DIR)
print(f"    OK  {CHECKPOINT_DIR} (writable)")

print("\nAll pre-flight checks passed.")

## 5. Load Notebook 1's split (read-only) and confirm it is unmodified

Same MD5-hash-based proof pattern as Notebook 2: the hash is recorded now
and re-checked at the very end of this notebook (Section 19) to prove
`split_manifest.csv` was never written to by this notebook.

In [ ]:
def file_md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

split_manifest_md5_before = file_md5(split_manifest_path)
manifest_df = pd.read_csv(split_manifest_path)
print("split_manifest.csv MD5 (recorded at load time):", split_manifest_md5_before)
print(f"Loaded split_manifest.csv: {len(manifest_df)} rows")

EXPECTED_SPLIT = {
    "total": 2468, "normal": 1925, "glaucoma": 543,
    "train": 1729, "val": 370, "test": 369,
}
actual = {
    "total": len(manifest_df),
    "normal": int((manifest_df["label"] == 0).sum()),
    "glaucoma": int((manifest_df["label"] == 1).sum()),
    "train": int((manifest_df["split"] == "train").sum()),
    "val": int((manifest_df["split"] == "val").sum()),
    "test": int((manifest_df["split"] == "test").sum()),
}
print("\nRegression check against the approved Notebook 1 dataset:")
all_match = True
for k in EXPECTED_SPLIT:
    ok = actual[k] == EXPECTED_SPLIT[k]
    all_match &= ok
    print(f"  [{'PASS' if ok else 'MISMATCH'}] {k:10s}: actual={actual[k]:5d}  expected={EXPECTED_SPLIT[k]:5d}")
assert all_match, "split_manifest.csv does not match the approved Notebook 1 dataset."
print("\nOverall: MATCHES the approved Notebook 1 split.")

## 6. Build DataLoaders (Notebook 2's pipeline, unmodified)

Uses `preprocessing.get_dataloaders`, i.e. exactly the CLAHE + augmentation
+ normalization pipeline verified in Notebook 2 -- nothing about the
transforms is redefined here.

In [ ]:
train_loader, val_loader, test_loader, train_ds, val_ds, test_ds = pp.get_dataloaders(
    manifest_df, DATASET_OUT_DIR, batch_size=BATCH_SIZE, num_workers=0, seed=SEED
)

print(f"train_ds: {len(train_ds)} images  ({len(train_loader)} batches @ batch_size={BATCH_SIZE})")
print(f"val_ds  : {len(val_ds)} images  ({len(val_loader)} batches @ batch_size={BATCH_SIZE})")
print(f"test_ds : {len(test_ds)} images  ({len(test_loader)} batches @ batch_size={BATCH_SIZE})")

assert len(train_ds) == EXPECTED_SPLIT["train"]
assert len(val_ds) == EXPECTED_SPLIT["val"]
assert len(test_ds) == EXPECTED_SPLIT["test"]

_images, _labels = next(iter(train_loader))
print(f"\nSample train batch: images={tuple(_images.shape)} labels={tuple(_labels.shape)} "
      f"label values={sorted(_labels.unique().tolist())}")
assert tuple(_images.shape[1:]) == (3, 224, 224)
print("PASS: DataLoaders built and verified against the approved split.")

## 7. Class weights

Same `compute_class_weights` function from `preprocessing.py` (Notebook 2),
computed from the training split's actual counts.

In [ ]:
class_weights, train_counts = pp.compute_class_weights(manifest_df, split="train")
class_weights = class_weights.to(device)

print("Train split class counts:", {CLASS_MAPPING[k]: v for k, v in train_counts.items()})
print("Class weights (index 0=Normal, 1=Glaucoma):", class_weights.tolist())
print("(weight[1] > weight[0] as expected -- Glaucoma is the minority class in training.)")

## 8. Build the ResNet50 model

Loads `torchvision.models.resnet50` with `ResNet50_Weights.IMAGENET1K_V1`
pretrained weights (the original/"classic" ResNet50 ImageNet weights --
chosen over the newer `IMAGENET1K_V2` recipe so results are comparable to
the ResNet50 baseline most commonly cited in prior literature, rather than a
newer training recipe with different tricks baked in). The final fully
connected layer is replaced with a fresh 2-class linear layer (randomly
initialized, since the task -- glaucoma vs. normal -- has no correspondence
to any of the 1,000 ImageNet classes).

`freeze_backbone` / `unfreeze_for_phase_b` set `requires_grad` by parameter
name -- they don't rebuild the model, so the same `model` object is reused
across both phases (Phase B explicitly reloads Phase A's *best* checkpoint
weights first, per Section 12, rather than assuming the in-memory model
already holds them).

In [ ]:
def build_resnet50(num_classes=2, pretrained=True):
    weights = ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
    model = resnet50(weights=weights)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

def freeze_backbone(model):
    """Phase A: only the FC head is trainable."""
    for name, param in model.named_parameters():
        param.requires_grad = name.startswith("fc.")

def unfreeze_for_phase_b(model):
    """Phase B: layer3, layer4, and fc are trainable; conv1/bn1/layer1/layer2 stay frozen."""
    for name, param in model.named_parameters():
        param.requires_grad = (
            name.startswith("layer3.") or name.startswith("layer4.") or name.startswith("fc.")
        )

def count_trainable(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

model = build_resnet50(num_classes=2, pretrained=True)
model.to(device)
print("ResNet50 built with ImageNet1K_V1 pretrained weights, fc replaced with Linear(2048, 2).")
print("fc layer:", model.fc)

## 9. Generic training/validation loop (shared by both phases)

`run_training_phase` trains for up to `max_epochs`, applies
`ReduceLROnPlateau(patience=3, factor=0.5)` and early stopping
(`patience=6`) -- **both monitoring validation loss**, the same metric used
for checkpoint selection (Section 8's design note / Section 13's model
selection), so "best" means the same thing everywhere in this notebook. The
scheduler and early-stopping counters are fresh for each phase (Phase B is
a structurally different optimization problem -- new parameter groups, new
learning rates -- so carrying over Phase A's counters would not be
meaningful). Only validation data is used anywhere in this function -- the
test set is never referenced.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def run_training_phase(model, train_loader, val_loader, criterion, optimizer, device,
                        max_epochs, scheduler_patience, scheduler_factor, early_stop_patience,
                        checkpoint_path, phase_name):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=scheduler_patience, factor=scheduler_factor
    )
    best_val_loss = float("inf")
    epochs_no_improve = 0
    history = {"epoch": [], "train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}

    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        scheduler.step(val_loss)
        current_lr = [g["lr"] for g in optimizer.param_groups]

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(current_lr)

        improved = val_loss < best_val_loss
        if improved:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save({
                "phase": phase_name, "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss, "val_acc": val_acc,
                "class_mapping": CLASS_MAPPING,
            }, checkpoint_path)
        else:
            epochs_no_improve += 1

        print(f"[{phase_name}] Epoch {epoch:2d}/{max_epochs} "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} lr={current_lr} "
              f"{'* saved' if improved else f'(no improve {epochs_no_improve}/{early_stop_patience})'} "
              f"({time.time()-t0:.1f}s)")

        if epochs_no_improve >= early_stop_patience:
            print(f"[{phase_name}] Early stopping at epoch {epoch} "
                  f"(no val_loss improvement for {early_stop_patience} epochs).")
            break

    return history, best_val_loss

print("Training loop functions defined: train_one_epoch, evaluate, run_training_phase.")

## 10. Phase A — frozen backbone, train the FC head

`freeze_backbone` sets every parameter's `requires_grad` except `fc.*`. The
optimizer only receives `model.fc.parameters()` -- passing frozen params to
an optimizer would waste memory on unused momentum buffers and is a common
source of confusing bugs, so this is deliberate, not just "won't update
because requires_grad=False".

In [ ]:
freeze_backbone(model)
trainable, total = count_trainable(model)
print(f"Phase A trainable parameters: {trainable:,} / {total:,} ({trainable/total:.2%})")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer_a = torch.optim.Adam(model.fc.parameters(), lr=PHASE_A_LR)

print(f"Optimizer: Adam(model.fc.parameters(), lr={PHASE_A_LR})")
print(f"Loss: CrossEntropyLoss(weight={class_weights.tolist()})")
print()

history_a, best_val_loss_a = run_training_phase(
    model, train_loader, val_loader, criterion, optimizer_a, device,
    max_epochs=PHASE_A_MAX_EPOCHS, scheduler_patience=SCHEDULER_PATIENCE,
    scheduler_factor=SCHEDULER_FACTOR, early_stop_patience=EARLY_STOP_PATIENCE,
    checkpoint_path=PHASE_A_CKPT, phase_name="Phase A",
)

print(f"\nPhase A complete. Best val_loss={best_val_loss_a:.4f}, checkpoint saved to {PHASE_A_CKPT}")

with open(os.path.join(CHECKPOINT_DIR, "phaseA_history.json"), "w") as f:
    json.dump(history_a, f, indent=2)

## 11. Phase B — fine-tune `layer3`, `layer4`, `fc`

First, **explicitly reload Phase A's best checkpoint from disk** (not just
whatever is currently in memory -- if early stopping triggered, the
in-memory weights are from the *last* epoch, not necessarily the *best*
one). Then unfreeze `layer3`/`layer4`/`fc` and build a param-group optimizer
with a lower backbone LR (`1e-4`) than FC LR (`1e-3`), so the pretrained
features are nudged gently while the head can still move quickly.

In [ ]:
_ckpt_a = torch.load(PHASE_A_CKPT, map_location=device)
model.load_state_dict(_ckpt_a["model_state_dict"])
print(f"Loaded Phase A best checkpoint (epoch {_ckpt_a['epoch']}, val_loss={_ckpt_a['val_loss']:.4f}) "
      f"before starting Phase B.")

unfreeze_for_phase_b(model)
trainable, total = count_trainable(model)
print(f"Phase B trainable parameters: {trainable:,} / {total:,} ({trainable/total:.2%})")

backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("fc.")]
fc_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("fc.")]

optimizer_b = torch.optim.Adam([
    {"params": backbone_params, "lr": PHASE_B_BACKBONE_LR},
    {"params": fc_params, "lr": PHASE_B_FC_LR},
])
print(f"Optimizer: Adam([{{backbone: lr={PHASE_B_BACKBONE_LR}}}, {{fc: lr={PHASE_B_FC_LR}}}])")
print(f"  backbone param tensors: {len(backbone_params)} | fc param tensors: {len(fc_params)}")

criterion_b = nn.CrossEntropyLoss(weight=class_weights)  # same weighted loss as Phase A
print()

history_b, best_val_loss_b = run_training_phase(
    model, train_loader, val_loader, criterion_b, optimizer_b, device,
    max_epochs=PHASE_B_MAX_EPOCHS, scheduler_patience=SCHEDULER_PATIENCE,
    scheduler_factor=SCHEDULER_FACTOR, early_stop_patience=EARLY_STOP_PATIENCE,
    checkpoint_path=PHASE_B_CKPT, phase_name="Phase B",
)

print(f"\nPhase B complete. Best val_loss={best_val_loss_b:.4f}, checkpoint saved to {PHASE_B_CKPT}")

with open(os.path.join(CHECKPOINT_DIR, "phaseB_history.json"), "w") as f:
    json.dump(history_b, f, indent=2)

## 12. Final model selection: Phase A vs. Phase B (validation loss only)

**Confirmed decision:** the final selected model is whichever of Phase A's
best checkpoint and Phase B's best checkpoint has the lower validation
loss -- Phase B fine-tuning is not assumed to win. This comparison uses
`val_loss` only; the test set has not been touched anywhere above and is
still untouched here.

In [ ]:
_ckpt_a = torch.load(PHASE_A_CKPT, map_location=device)
_ckpt_b = torch.load(PHASE_B_CKPT, map_location=device)

print(f"Phase A best: epoch={_ckpt_a['epoch']}, val_loss={_ckpt_a['val_loss']:.4f}, val_acc={_ckpt_a['val_acc']:.4f}")
print(f"Phase B best: epoch={_ckpt_b['epoch']}, val_loss={_ckpt_b['val_loss']:.4f}, val_acc={_ckpt_b['val_acc']:.4f}")

if _ckpt_a["val_loss"] <= _ckpt_b["val_loss"]:
    final_ckpt, final_source = _ckpt_a, "Phase A"
else:
    final_ckpt, final_source = _ckpt_b, "Phase B"

print(f"\nFinal selected model: {final_source} "
      f"(val_loss={final_ckpt['val_loss']:.4f} vs. the other phase's "
      f"{(_ckpt_b if final_source == 'Phase A' else _ckpt_a)['val_loss']:.4f})")

final_model = build_resnet50(num_classes=2, pretrained=False)
final_model.load_state_dict(final_ckpt["model_state_dict"])
final_model.to(device)
final_model.eval()

torch.save({**final_ckpt, "selected_from": final_source}, FINAL_CKPT)
print(f"Saved final selected checkpoint to: {FINAL_CKPT}")

## 13. Training/validation loss and accuracy curves (both phases)

A single figure, two panels (loss, accuracy), both phases plotted with a
vertical line marking the Phase A -> Phase B transition, so the effect of
unfreezing `layer3`/`layer4` is visible directly.

In [ ]:
epochs_a = history_a["epoch"]
epochs_b_offset = [e + len(epochs_a) for e in history_b["epoch"]]
transition_x = len(epochs_a) + 0.5

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_a, history_a["train_loss"], label="Phase A train", color="tab:blue")
axes[0].plot(epochs_a, history_a["val_loss"], label="Phase A val", color="tab:blue", linestyle="--")
axes[0].plot(epochs_b_offset, history_b["train_loss"], label="Phase B train", color="tab:orange")
axes[0].plot(epochs_b_offset, history_b["val_loss"], label="Phase B val", color="tab:orange", linestyle="--")
axes[0].axvline(transition_x, color="gray", linestyle=":", label="Phase A -> B")
axes[0].set_xlabel("Epoch (concatenated across phases)")
axes[0].set_ylabel("Loss (weighted CrossEntropy)")
axes[0].set_title("Loss")
axes[0].legend(fontsize=8)

axes[1].plot(epochs_a, history_a["train_acc"], label="Phase A train", color="tab:blue")
axes[1].plot(epochs_a, history_a["val_acc"], label="Phase A val", color="tab:blue", linestyle="--")
axes[1].plot(epochs_b_offset, history_b["train_acc"], label="Phase B train", color="tab:orange")
axes[1].plot(epochs_b_offset, history_b["val_acc"], label="Phase B val", color="tab:orange", linestyle="--")
axes[1].axvline(transition_x, color="gray", linestyle=":", label="Phase A -> B")
axes[1].set_xlabel("Epoch (concatenated across phases)")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy")
axes[1].legend(fontsize=8)

plt.tight_layout()
curves_path = os.path.join(FIGURES_OUT_DIR, "notebook3_training_curves.png")
fig.savefig(curves_path, dpi=130)
plt.show()
print("Saved:", curves_path)

with open(os.path.join(CHECKPOINT_DIR, "training_history.json"), "w") as f:
    json.dump({"phase_a": history_a, "phase_b": history_b,
                "final_selected_from": final_source}, f, indent=2)
print("Saved combined training history JSON.")

## 14. Run the final selected model on the test set — exactly once

This is the **only** cell in this notebook that touches `test_loader`.
Nothing above (Phase A, Phase B, scheduler, early stopping, or the Phase
A/B selection in Section 12) used the test set in any way.

In [ ]:
@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    for images, labels in loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, POSITIVE_CLASS]
        preds = outputs.argmax(dim=1)
        all_labels.extend(labels.numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

test_labels, test_preds, test_probs = get_predictions(final_model, test_loader, device)
print(f"Ran inference on {len(test_labels)} test images using the final selected model ({final_source}).")
print("Label distribution in test set:", {CLASS_MAPPING[k]: int((test_labels == k).sum()) for k in (0, 1)})

## 15. Test-set metrics

Precision/Recall/F1/ROC-AUC/Sensitivity/Specificity are all computed with
**Glaucoma (label 1) as the positive class** -- the clinically relevant
choice for a detection task (sensitivity = correctly caught glaucoma
cases; specificity = correctly cleared normal cases). No target
accuracy/AUC was assumed anywhere in this notebook -- these are simply
whatever the selected model achieved.

In [ ]:
accuracy = accuracy_score(test_labels, test_preds)
precision = precision_score(test_labels, test_preds, pos_label=POSITIVE_CLASS, zero_division=0)
recall_sensitivity = recall_score(test_labels, test_preds, pos_label=POSITIVE_CLASS, zero_division=0)
f1 = f1_score(test_labels, test_preds, pos_label=POSITIVE_CLASS, zero_division=0)
roc_auc = roc_auc_score(test_labels, test_probs)

cm = confusion_matrix(test_labels, test_preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

print("=== Test set metrics (final selected model: %s) ===" % final_source)
print(f"Accuracy            : {accuracy:.4f}")
print(f"Precision (Glaucoma): {precision:.4f}")
print(f"Recall/Sensitivity  : {recall_sensitivity:.4f}")
print(f"Specificity         : {specificity:.4f}")
print(f"F1 (Glaucoma)       : {f1:.4f}")
print(f"ROC-AUC             : {roc_auc:.4f}")
print(f"\nConfusion matrix (rows=true, cols=pred, order=[Normal, Glaucoma]):")
print(cm)
print(f"TN={tn} FP={fp} FN={fn} TP={tp}")

metrics = {
    "final_selected_from": final_source,
    "final_checkpoint_path": FINAL_CKPT,
    "positive_class": CLASS_MAPPING[POSITIVE_CLASS],
    "n_test": int(len(test_labels)),
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall_sensitivity": float(recall_sensitivity),
    "specificity": float(specificity),
    "f1": float(f1),
    "roc_auc": float(roc_auc),
    "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    "phase_a_best_val_loss": float(_ckpt_a["val_loss"]),
    "phase_b_best_val_loss": float(_ckpt_b["val_loss"]),
}
metrics_path = os.path.join(RESULTS_OUT_DIR, "notebook3_resnet50_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print("\nSaved metrics JSON to:", metrics_path)

## 16. Confusion matrix figure

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Normal", "Glaucoma"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Normal", "Glaucoma"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix -- Test Set ({final_source})")

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
cm_path = os.path.join(FIGURES_OUT_DIR, "notebook3_confusion_matrix.png")
fig.savefig(cm_path, dpi=130)
plt.show()
print("Saved:", cm_path)

## 17. ROC curve figure

In [ ]:
fpr, tpr, thresholds = roc_curve(test_labels, test_probs)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr, tpr, color="tab:blue", label=f"ResNet50 ({final_source}), AUC={roc_auc:.4f}")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", label="Chance (AUC=0.5)")
ax.set_xlabel("False Positive Rate (1 - Specificity)")
ax.set_ylabel("True Positive Rate (Sensitivity)")
ax.set_title("ROC Curve -- Test Set")
ax.legend(loc="lower right")
plt.tight_layout()
roc_path = os.path.join(FIGURES_OUT_DIR, "notebook3_roc_curve.png")
fig.savefig(roc_path, dpi=130)
plt.show()
print("Saved:", roc_path)

## 18. Final verification summary

Re-checks that `split_manifest.csv` was never modified (MD5 hash unchanged
from Section 5), then prints a consolidated summary of everything this
notebook did.

In [ ]:
split_manifest_md5_after = file_md5(split_manifest_path)
split_manifest_unchanged = (split_manifest_md5_before == split_manifest_md5_after)

print("=" * 60)
print("NOTEBOOK 3 -- FINAL VERIFICATION SUMMARY")
print("=" * 60)
print(f"split_manifest.csv MD5 unchanged : {split_manifest_unchanged}")
print(f"Device used for training         : {device}")
print(f"Phase A: {len(history_a['epoch'])} epochs run, best val_loss={best_val_loss_a:.4f}")
print(f"Phase B: {len(history_b['epoch'])} epochs run, best val_loss={best_val_loss_b:.4f}")
print(f"Final selected model             : {final_source}")
print(f"Test set size                    : {len(test_labels)} (touched exactly once, in Section 14)")
print(f"Accuracy / Precision / Recall / Specificity / F1 / ROC-AUC:")
print(f"  {accuracy:.4f} / {precision:.4f} / {recall_sensitivity:.4f} / {specificity:.4f} / {f1:.4f} / {roc_auc:.4f}")
print(f"Checkpoints saved  : {PHASE_A_CKPT}")
print(f"                     {PHASE_B_CKPT}")
print(f"                     {FINAL_CKPT}")
print(f"Metrics JSON       : {metrics_path}")
print(f"Figures saved at   : {FIGURES_OUT_DIR}")
print("=" * 60)

assert split_manifest_unchanged, "split_manifest.csv was modified -- this must never happen in Notebook 3."
print("\nALL CHECKS PASSED. Notebook 3 is verified and did not alter Notebook 1's split.")